<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c37-header.png" alt="Nextia Learning · Recommendation and Personalisation: Show Each Person What Matters" width="100%">

# Privacy and sensitive suggestions

**[Privacy and sensitive suggestions](https://learning.nextia-ai.com/courses/recommendations/m09/privacy-and-sensitive-suggestions/)** · Recommendation and Personalisation: Show Each Person What Matters · Module 9, lesson 3 of 3 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** find the personal data a recommender does not need, apply a deletion request to the model as well as the data, and keep sensitive products out of a shared family inbox.

| | |
|---|---|
| **Time** | About 30 minutes |
| **Needs** | An internet connection for the setup cells (about 8 MB). No account and no key. |
| **You should know** | Monitoring and retraining, from [Monitor and retrain](https://learning.nextia-ai.com/courses/recommendations/m09/monitor-and-retrain/). |
| **Tested** | Python 3.12 with NumPy 2.5.4, pandas 3.1.1, SciPy 1.18.1 and matplotlib 3.11.2 (implicit 0.7.3 or LightGBM 4.7.0 where the lesson needs it), in Jupyter on a laptop CPU (Apple M4, 16 GB), from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/recommendations/m09/privacy-and-sensitive-suggestions/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C37/M09-L03-privacy-and-sensitive-suggestions/privacy-and-sensitive-suggestions-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs the versions that the course uses: numpy 2.5.4, pandas 3.1.1, scipy 1.18.1, and matplotlib for the pictures. It needs an internet connection and takes about a minute the first time.

You should see one line per package, with the versions above. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again.

In [ ]:
%pip install -q --disable-pip-version-check numpy==2.5.4 pandas==3.1.1 scipy==1.18.1 matplotlib==3.11.2
import warnings; warnings.filterwarnings("ignore")
from importlib.metadata import version
for name in ['numpy', 'pandas', 'scipy']:
    print(name, version(name))

## Setup: get the Larkfield data

Run the next cell. It downloads the files that this lesson needs (about 8 MB) into a folder `larkfield-recs` and checks each file's checksum. It needs an internet connection and no account, and takes about 20 seconds.

You should see `Ready:` and the number of files. If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `larkfield-recs` (in Colab: the **Files** panel on the left), then run the cell again.

Where the data comes from: Larkfield is an invented shop, and its shop log is **synthetic**: our code made the products, the customers and every view, basket and order, with a model of customer behaviour. The [dataset card](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C37/data/dataset-card.md) explains how.

In [ ]:
# Setup: download the files into larkfield-recs/ and check them.
import hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

LABS = "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C37/"
FILES = {
    "data/products.csv": "3da37a3809f6067da12eb5778b8abcf809545599db6959208c72a790861407ff",
    "data/customers.csv": "282446db7097aec2b30c27c574a6c7b13a8cffce8053fed212d80bf86cc1411c",
    "data/events.csv.gz": "3f46ddfff683586ab1548bc0e6756d85a439a1de941e6de3030edc383d06d1b3",
}
HERE = Path.cwd()
# The notebook works in its own folder, marked with a file, so it never
# writes into your own project if you open the notebook there.
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
PROJECT = Path("larkfield-recs").resolve()
(PROJECT / "data").mkdir(parents=True, exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

for name, expected in FILES.items():
    path = PROJECT / name
    path.parent.mkdir(parents=True, exist_ok=True)
    if not (path.exists() and sha256(path) == expected):
        local = HERE / name  # a copy next to the notebook, if there is one
        if local.exists():
            shutil.copy(local, path)
        else:
            download(LABS + name, path)
        if sha256(path) != expected:
            raise SystemExit(f"{name} has the wrong checksum. Delete the folder larkfield-recs and run again.")

os.chdir(PROJECT)  # work in the project folder, as in the lesson
print(f"Ready: {len(FILES)} files in larkfield-recs.")

## Setup: helpers

The next cell loads NumPy as `np`, pandas as `pd`, SciPy's sparse matrices as `sp` and matplotlib. It defines the course's small helpers, the same ideas as the course project:

- `products()`, `customers()` and `events()` read the data. `events()` has one row per view, basket add or order, with the column `day` (days since 1 February 2025).
- `split(period)` gives the events before the period (`train`, to learn from) and, for each customer who ordered in the period, the products they ordered (`truth`). The periods are `"validation"`, `"test"` and `"launch"`.
- `recall_at_k(list, ordered, k)`, `ndcg_at_k(list, ordered, k)` and `coverage(lists, n_products)` measure suggestion lists.
- `show(ids)` prints products with their title and subcategory.

It also defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.

Run the cell. It prints nothing.

In [ ]:
# Helpers: the data, the time split, the measures, and checks that do not show the answer.
import hashlib, json, warnings
import numpy as np
import pandas as pd
import scipy.sparse as sp
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
START = pd.Timestamp("2025-02-01")

def products():
    """One row per product (2,994)."""
    df = pd.read_csv("data/products.csv", keep_default_na=False)
    for col in ("age_restricted", "sensitive", "in_stock"):
        df[col] = df[col].astype(str).eq("True")
    return df

def customers():
    """One row per customer (40,000)."""
    return pd.read_csv("data/customers.csv")

def events():
    """One row per event: view, basket or order. day = days since 1 February 2025."""
    ev = pd.read_csv("data/events.csv.gz", parse_dates=["timestamp"])
    ev["day"] = (ev["timestamp"] - START).dt.days
    return ev

def split(period, ev=None):
    """train = the events before the period; truth = {customer: set of products ordered in the period}."""
    ev = events() if ev is None else ev
    first, last = json.load(open("data/splits.json"))[period]
    a, b = (pd.Timestamp(first) - START).days, (pd.Timestamp(last) - START).days
    train = ev[ev["day"] < a]
    target = ev[(ev["day"] >= a) & (ev["day"] <= b) & (ev["event"] == "order")]
    truth = target.groupby("customer_id")["product_id"].agg(set).to_dict()
    return train, truth

def recall_at_k(rec, ordered, k=10):
    """Share of the ordered products that are in the first k suggestions."""
    return len(set(list(rec)[:k]) & set(ordered)) / len(ordered)

def ndcg_at_k(rec, ordered, k=10):
    """DCG of the list divided by the best possible DCG (discount log2(place + 1))."""
    gains = [1.0 if p in ordered else 0.0 for p in list(rec)[:k]]
    dcg = sum(g / np.log2(i + 2) for i, g in enumerate(gains))
    ideal = sum(1 / np.log2(i + 2) for i in range(min(len(ordered), k)))
    return dcg / ideal

def coverage(lists, n_products, k=10):
    """Share of the catalogue that appears in at least one list."""
    return len({p for rec in lists for p in list(rec)[:k]}) / n_products

def rounded(x, n=3):
    """x rounded to n places (None or ... unchanged: the task is not done yet)."""
    return x if x is None or x is ... else round(float(x), n)

def show(ids, n=10):
    """Print products: place, ID, title, subcategory."""
    global _PR
    if "_PR" not in globals():
        _PR = products().set_index("product_id")
    for i, p in enumerate(list(ids)[:n], 1):
        print(f"{i:2}. {p}  {_PR.loc[p, 'title'][:52]:<52}  {_PR.loc[p, 'subcategory']}")

def fingerprint(value):
    def norm(v):
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 3)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
        return
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. Which personal data does a recommender need?

**Data minimisation** means keeping only the personal data that a purpose needs. Run the next cell. It lists the columns of the two files about people.

In [ ]:
cu = customers()
print("customers.csv columns:", list(cu.columns))
print("events columns:       ", list(events().columns))
print(f"share an email address with family: {cu['shared_email'].sum():,}; opted out of personal suggestions: {cu['opted_out'].sum():,}")

The shop log has no names, no email addresses and no payment details: the recommender does not need them. It needs who did what, and when. Look at `region`: the course's project never uses it. A column that no method uses is a column you could stop collecting for this purpose.

## 2. A deletion request must reach the model

The next cell trains a small item-item model (people who touched this product also touched…) on the whole log. It takes a few seconds.

In [ ]:
ev = events()
cust = np.sort(ev["customer_id"].unique())
prod = np.sort(products()["product_id"].unique())

def similarity(ev, neighbours=50):
    """Products x products: cosine of the customers who touched both (any event, counted once), top 50 per product."""
    ci = np.searchsorted(cust, ev["customer_id"]); pi = np.searchsorted(prod, ev["product_id"])
    B = sp.csr_matrix((np.ones(len(ev), dtype=np.float32), (ci, pi)), shape=(len(cust), len(prod)))
    B.data[:] = 1
    B.sum_duplicates(); B.data[:] = 1
    n = np.asarray(B.sum(axis=0)).ravel()
    C = (B.T @ B).tocoo()
    keep = C.row != C.col
    r, c, d = C.row[keep], C.col[keep], C.data[keep]
    S = sp.csr_matrix((d / np.sqrt(n[r] * n[c]), (r, c)), shape=(len(prod), len(prod)))
    rows, cols, vals = [], [], []
    for i in range(S.shape[0]):                         # keep the strongest neighbours of each product
        a, b = S.indptr[i], S.indptr[i + 1]
        top = np.argsort(-S.data[a:b], kind="stable")[:neighbours]
        rows += [i] * len(top); cols += S.indices[a:b][top].tolist(); vals += S.data[a:b][top].tolist()
    return sp.csr_matrix((vals, (rows, cols)), shape=S.shape)

before = similarity(ev)
print(f"item-item model: {before.nnz:,} product pairs")

## 3. Your turn: what does one customer change?

Customer C100123 asks Larkfield to delete their data. Remove their events, train the model again, and count the products whose neighbours changed.

> **Predict.** One customer of 40,000: does their deletion change the model at all?

In [ ]:
who = "C100123"
mine = ev["customer_id"] == who
print(f"{who}: {mine.sum()} events")
after = similarity(ev[~mine])                           # trained again without this customer
diff = abs(after - before)
changed = ...        # how many products (rows) have at least one different value? Hint: (diff > 1e-6).sum(axis=1)
print(changed)

In [ ]:
expect('changed', changed, 391)

One customer with 22 events changes the neighbours of 391 products. Deleting their rows from the data is not enough: a model trained before the request still holds what they did, until it is trained again. In the project, `recs forget C100123` does both, and rebuilds the saved lists. It prints 394 products for the same customer: its model is built in a slightly different way, so the count differs a little. The lesson is the same.

## 4. Sensitive products and a shared inbox

Some customers share one email address with their family. A suggestion for a mobility aid or a pregnancy test in that inbox tells the family something private. The next cell builds an email list for such a customer from a constructed personal list.

In [ ]:
pr = products().set_index("product_id")
sensitive = set(pr.index[pr["sensitive"]])
shared = set(cu.loc[cu["shared_email"], "customer_id"])

def email_list(customer_id, personal_list, k=10):
    """Never a sensitive product in an inbox that a family shares."""
    if customer_id in shared:
        personal_list = [p for p in personal_list if p not in sensitive]
    return personal_list[:k]

# a constructed list for a customer who shares an email address: five mobility aids and five glasses
mobility = list(pr.index[pr["subcategory"] == "Mobility aids"][:5])
glasses = list(pr.index[pr["subcategory"] == "Glasses"][:10])
home = mobility + glasses[:5]
customer = sorted(shared)[0]
print("home page:", len(set(home) & sensitive), "sensitive of", len(home))
mail = email_list(customer, home + glasses[5:])
print("email:    ", len(set(mail) & sensitive), "sensitive of", len(mail))
show(mail, 3)

> **Check.** 3,178 customers share an email address. Their emails never contain a sensitive product, whatever the personal list says. The project's service does the same with `surface=email`.

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Recommend for an online retailer](https://learning.nextia-ai.com/courses/recommendations/m10/recommend-for-an-online-retailer/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/recommendations/m09/privacy-and-sensitive-suggestions/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The Larkfield shop log is synthetic: our code made the products, the customers and everything they did, with a model of customer behaviour. No real customer is in this data.